# Walk-Forward Regime Momentum

Re-estimates the volatility-regime momentum model in chronological windows.

This notebook downloads daily historical market prices from Yahoo Finance. Results depend on the configured dates and symbols.


In [ ]:
"""
ENHANCED VOLATILITY REGIME STRATEGY - WALK-FORWARD PRODUCTION VERSION
=====================================================================

CRITICAL ANTI-LOOK-AHEAD IMPLEMENTATION:
----------------------------------------
1. Regime thresholds recalculated using EXPANDING WINDOW only
2. Walk-forward validation with periodic threshold refitting
3. Portfolio optimization uses only historical data up to T-1
4. All features and signals use strict T-1 to T flow
5. Comprehensive fold tracking for validation

WALK-FORWARD METHODOLOGY:
------------------------
- Minimum training period: 252 days (1 year)
- Threshold refit frequency: 63 days (quarterly)
- Expanding window: Uses ALL historical data up to current point
- No forward-filling or future data contamination

REGIME ALLOCATION STRATEGY:
--------------------------
Regime 0 - Low Vol Downtrend:    HRP (80% exposure)
Regime 1 - Low Vol Uptrend:      Max Sharpe (120% exposure)
Regime 2 - Normal Vol Downtrend: BEARISH HEDGE (7.5% + 92.5% Cash)
Regime 3 - Normal Vol Uptrend:   HRP (80% exposure)
Regime 4 - High Vol Downtrend:   BEARISH HEDGE (7.5% + 92.5% Cash)
Regime 5 - High Vol Uptrend:     Max Sharpe (120% exposure)
"""

from _market_data import download_price_records
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from typing import Tuple, Dict, List, Optional
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from dataclasses import dataclass
import warnings

warnings.filterwarnings("ignore")


@dataclass
class RegimeThresholds:
    """Stores regime thresholds from training period."""

    median_vol: float
    percentile_75_vol: float
    training_end_date: pd.Timestamp
    training_size: int


class WalkForwardRegimeDetector:
    """
    WALK-FORWARD REGIME DETECTOR WITH NO LOOK-AHEAD BIAS
    ====================================================

    ANTI-LOOK-AHEAD GUARANTEE:
    - Thresholds calculated ONLY from expanding historical window
    - Periodic refitting as new data becomes available
    - All volatility calculations use T-1 data only
    - Regime prediction uses frozen thresholds from training

    WALK-FORWARD PROCESS:
    1. Calculate features using T-1 data only (double-lagged)
    2. Fit thresholds on expanding window of historical data
    3. Predict regime using frozen thresholds
    4. Refit thresholds every REFIT_FREQUENCY days
    5. Track all folds for validation
    """

    def __init__(
        self, vol_lookback: int = 20, min_training_days: int = 252, refit_frequency: int = 63
    ):
        """
        Initialize walk-forward regime detector.

        Args:
            vol_lookback: Rolling window for volatility calculation (default: 20 days)
            min_training_days: Minimum days before first prediction (default: 252 = 1 year)
            refit_frequency: Days between threshold refitting (default: 63 = quarterly)
        """
        self.vol_lookback = vol_lookback
        self.min_training_days = min_training_days
        self.refit_frequency = refit_frequency
        self.regime_names = [
            "Low Vol - Downtrend",
            "Low Vol - Uptrend",
            "Normal Vol - Downtrend",
            "Normal Vol - Uptrend",
            "High Vol - Downtrend",
            "High Vol - Uptrend",
        ]
        self.current_thresholds = None
        self.fold_history = []

    def calculate_features_t_minus_1(self, price_data: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate rolling volatility and returns using STRICT T-1 methodology.

        CRITICAL ANTI-LOOK-AHEAD DESIGN:
        - Returns calculated from Close[T-1]/Close[T-2]
        - Returns lagged once for volatility calculation (ensures T-2 returns for vol at T-1)
        - Volatility lagged again to ensure T-1 availability at time T
        - Double-lagging prevents ANY forward contamination

        Args:
            price_data: DataFrame with ['date', 'close'] columns

        Returns:
            DataFrame with double-lagged features
        """
        df = price_data.copy()
        df = df.sort_values("date").reset_index(drop=True)
        df["log_return"] = np.log(df["close"] / df["close"].shift(1))
        df["log_return_for_vol"] = df["log_return"].shift(1)
        df["rolling_vol"] = df["log_return_for_vol"].rolling(
            window=self.vol_lookback, min_periods=self.vol_lookback
        ).std() * np.sqrt(252)
        df["rolling_mean"] = (
            df["log_return_for_vol"]
            .rolling(window=self.vol_lookback, min_periods=self.vol_lookback)
            .mean()
            * 252
        )
        df["rolling_vol_lag"] = df["rolling_vol"].shift(1)
        df["rolling_mean_lag"] = df["rolling_mean"].shift(1)
        return df

    def calculate_thresholds_expanding_window(
        self, features_df: pd.DataFrame, end_idx: int
    ) -> RegimeThresholds:
        """
        Calculate regime thresholds using EXPANDING WINDOW only.

        CRITICAL: This uses ONLY data from [0 : end_idx] to calculate thresholds.
        No future data is used.

        Args:
            features_df: DataFrame with lagged volatility features
            end_idx: End index for training window (exclusive)

        Returns:
            RegimeThresholds object with calculated thresholds
        """
        train_window = features_df.iloc[0:end_idx].copy()
        valid_vol = train_window["rolling_vol_lag"].dropna()
        if len(valid_vol) < 50:
            print(f"   WARNING: Insufficient data ({len(valid_vol)} points), using defaults")
            return RegimeThresholds(
                median_vol=0.12,
                percentile_75_vol=0.2,
                training_end_date=train_window["date"].iloc[-1],
                training_size=len(train_window),
            )
        median_vol = valid_vol.median()
        percentile_75_vol = valid_vol.quantile(0.75)
        return RegimeThresholds(
            median_vol=median_vol,
            percentile_75_vol=percentile_75_vol,
            training_end_date=train_window["date"].iloc[-1],
            training_size=len(train_window),
        )

    def predict_regime(
        self,
        rolling_vol_t_minus_1: float,
        rolling_mean_t_minus_1: float,
        thresholds: RegimeThresholds,
    ) -> Tuple[int, str]:
        """
        Predict regime using T-1 data and FROZEN thresholds from training.

        REGIME CLASSIFICATION:
        Volatility: Low (≤median) | Normal (median to 75th) | High (>75th)
        Trend: Downtrend (<0) | Uptrend (≥0)

        Args:
            rolling_vol_t_minus_1: Volatility from T-1
            rolling_mean_t_minus_1: Mean return from T-1
            thresholds: Frozen thresholds from training period

        Returns:
            Tuple of (regime_index, regime_name)
        """
        if pd.isna(rolling_vol_t_minus_1) or pd.isna(rolling_mean_t_minus_1):
            return (3, "Normal Vol - Uptrend")
        if rolling_vol_t_minus_1 <= thresholds.median_vol:
            vol_idx = 0
        elif rolling_vol_t_minus_1 <= thresholds.percentile_75_vol:
            vol_idx = 1
        else:
            vol_idx = 2
        trend_idx = 1 if rolling_mean_t_minus_1 >= 0 else 0
        regime_idx = vol_idx * 2 + trend_idx
        regime_name = self.regime_names[regime_idx]
        return (regime_idx, regime_name)

    def walk_forward_predict(self, features_df: pd.DataFrame) -> Tuple[pd.DataFrame, List[Dict]]:
        """
        Execute walk-forward regime prediction with NO LOOK-AHEAD BIAS.

        WALK-FORWARD PROCESS:
        --------------------
        For each prediction day T:

        TRAINING PHASE (every REFIT_FREQUENCY days):
        1. Use expanding window [0 : T] for threshold calculation
        2. Calculate median and 75th percentile volatility
        3. FREEZE these thresholds for next REFIT_FREQUENCY days

        PREDICTION PHASE (every day):
        4. Use T-1 volatility and trend data
        5. Apply FROZEN thresholds from training
        6. Predict regime for day T

        ITERATION:
        7. Move to next day
        8. Refit thresholds every REFIT_FREQUENCY days
        9. Continue until all days predicted

        Args:
            features_df: DataFrame with calculated features (must include lagged features)

        Returns:
            Tuple of (predictions_df, fold_list)
        """
        print("\n" + "=" * 80)
        print("WALK-FORWARD REGIME DETECTION (NO LOOK-AHEAD BIAS)")
        print("=" * 80)
        df_clean = features_df.dropna(subset=["rolling_vol_lag", "rolling_mean_lag"]).copy()
        df_clean = df_clean.reset_index(drop=True)
        print(f"Clean observations: {len(df_clean)}")
        print(f"Date range: {df_clean['date'].min().date()} to {df_clean['date'].max().date()}")
        df_clean["regime_idx"] = np.nan
        df_clean["regime_name"] = None
        df_clean["threshold_median"] = np.nan
        df_clean["threshold_75th"] = np.nan
        min_train = self.min_training_days
        refit_freq = self.refit_frequency
        print(f"\nWalk-forward configuration:")
        print(f"  Minimum training: {min_train} days (~{min_train / 252:.1f} years)")
        print(f"  Refit frequency: {refit_freq} days (~{refit_freq / 21:.1f} months)")
        print(f"  Predictions needed: {len(df_clean) - min_train}")
        fold_list = []
        fold_num = 0
        last_refit_idx = min_train
        current_thresholds = None
        for i in range(min_train, len(df_clean)):
            days_since_refit = i - last_refit_idx
            if current_thresholds is None or days_since_refit >= refit_freq:
                fold_num += 1
                train_start = 0
                train_end = i
                print(f"\n{'=' * 80}")
                print(f"FOLD {fold_num}: Refitting thresholds at index {i}")
                print(f"{'=' * 80}")
                print(f"Training window: [0:{train_end}] (EXPANDING WINDOW)")
                print(
                    f"Training dates: {df_clean['date'].iloc[0].date()} to {df_clean['date'].iloc[train_end - 1].date()}"
                )
                print(f"Training size: {train_end} days")
                current_thresholds = self.calculate_thresholds_expanding_window(df_clean, train_end)
                print(
                    f"Thresholds calculated from {current_thresholds.training_size} observations:"
                )
                print(
                    f"  Low/Normal Vol threshold: {current_thresholds.median_vol:.4f} ({current_thresholds.median_vol:.1%})"
                )
                print(
                    f"  Normal/High Vol threshold: {current_thresholds.percentile_75_vol:.4f} ({current_thresholds.percentile_75_vol:.1%})"
                )
                fold_list.append(
                    {
                        "fold_num": fold_num,
                        "train_start": train_start,
                        "train_end": train_end,
                        "train_start_date": df_clean["date"].iloc[train_start],
                        "train_end_date": df_clean["date"].iloc[train_end - 1],
                        "median_vol": current_thresholds.median_vol,
                        "percentile_75_vol": current_thresholds.percentile_75_vol,
                        "training_size": current_thresholds.training_size,
                    }
                )
                last_refit_idx = i
            vol_t_minus_1 = df_clean.loc[i, "rolling_vol_lag"]
            mean_t_minus_1 = df_clean.loc[i, "rolling_mean_lag"]
            regime_idx, regime_name = self.predict_regime(
                vol_t_minus_1, mean_t_minus_1, current_thresholds
            )
            df_clean.loc[i, "regime_idx"] = regime_idx
            df_clean.loc[i, "regime_name"] = regime_name
            df_clean.loc[i, "threshold_median"] = current_thresholds.median_vol
            df_clean.loc[i, "threshold_75th"] = current_thresholds.percentile_75_vol
        print(f"\n{'=' * 80}")
        print("WALK-FORWARD REGIME DETECTION COMPLETE")
        print(f"{'=' * 80}")
        print(f"Total folds executed: {len(fold_list)}")
        print(f"Predictions generated: {df_clean['regime_name'].notna().sum()}")
        regime_counts = df_clean["regime_name"].value_counts()
        print(f"\nRegime distribution:")
        for regime in self.regime_names:
            count = regime_counts.get(regime, 0)
            pct = count / len(df_clean) * 100 if len(df_clean) > 0 else 0
            print(f"  {regime:25s}: {count:4d} days ({pct:5.1f}%)")
        print("=" * 80)
        return (df_clean, fold_list)


class MaxSharpeOptimizer:
    """
    Maximum Sharpe Ratio Portfolio Optimizer (LONG-ONLY with leverage).

    FEATURES:
    - Ledoit-Wolf shrinkage for robust covariance estimation
    - Individual position size limits
    - Leverage capability up to max_exposure
    """

    def __init__(self, risk_free_rate: float = 0.02, max_exposure: float = 1.2):
        self.risk_free_rate = risk_free_rate
        self.max_exposure = max_exposure

    def optimize(self, returns: pd.DataFrame, max_weight: float = 0.15) -> Dict[str, float]:
        """
        Optimize for maximum Sharpe ratio.

        Args:
            returns: Historical returns DataFrame (assets as columns)
            max_weight: Maximum weight per asset (default: 15%)

        Returns:
            Dictionary mapping assets to optimal weights
        """
        if len(returns) < 60:
            return {}
        expected_returns = returns.mean() * 252
        cov_matrix = returns.cov() * 252
        try:
            lw = LedoitWolf()
            cov_shrunk, _ = (lw.fit(returns.values).covariance_, lw.shrinkage_)
            cov_matrix = pd.DataFrame(
                cov_shrunk * 252, index=returns.columns, columns=returns.columns
            )
        except:
            pass
        n_assets = len(expected_returns)

        def neg_sharpe(weights):
            port_return = np.dot(weights, expected_returns)
            port_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
            if port_vol == 0:
                return -np.inf
            return -(port_return - self.risk_free_rate) / port_vol

        constraints = [{"type": "eq", "fun": lambda x: np.sum(x) - self.max_exposure}]
        bounds = [(0, max_weight) for _ in range(n_assets)]
        x0 = np.array([self.max_exposure / n_assets] * n_assets)
        try:
            result = minimize(
                neg_sharpe,
                x0,
                method="SLSQP",
                bounds=bounds,
                constraints=constraints,
                options={"maxiter": 1000, "ftol": 1e-09},
            )
            if result.success:
                weights = result.x
                weights = np.where(weights < 0.01, 0, weights)
                if np.sum(weights) > 0:
                    weights = weights * self.max_exposure / np.sum(weights)
                    return {
                        asset: w for asset, w in zip(expected_returns.index, weights) if w > 0.01
                    }
        except:
            pass
        return {}


class HRPOptimizer:
    """
    Hierarchical Risk Parity (HRP) Optimizer.

    Lopez de Prado's algorithm:
    1. Hierarchical clustering based on correlation distance
    2. Quasi-diagonalization
    3. Recursive bisection for risk allocation
    """

    def __init__(self, linkage_method: str = "ward", target_exposure: float = 0.8):
        self.linkage_method = linkage_method
        self.target_exposure = target_exposure

    def optimize(self, returns: pd.DataFrame) -> Dict[str, float]:
        """
        Optimize using Hierarchical Risk Parity.

        Args:
            returns: Historical returns DataFrame

        Returns:
            Dictionary mapping assets to HRP weights
        """
        if len(returns) < 60:
            return {}
        try:
            corr_matrix = returns.corr().fillna(0)
            distance_matrix = np.sqrt(0.5 * (1 - corr_matrix))
            condensed_distances = squareform(distance_matrix.values)
            linkage_matrix = linkage(condensed_distances, method=self.linkage_method)
            sorted_indices = self._get_quasi_diagonal(linkage_matrix, len(corr_matrix))
            sorted_corr = corr_matrix.iloc[sorted_indices, sorted_indices]
            hrp_weights = self._get_recursive_bisection(sorted_corr)
            weight_dict = {}
            for i, asset in enumerate(corr_matrix.index):
                original_index = sorted_indices.index(i)
                weight = hrp_weights[original_index] * self.target_exposure
                if weight > 0.01:
                    weight_dict[asset] = weight
            return weight_dict
        except:
            return {}

    def _get_quasi_diagonal(self, linkage_matrix, num_assets):
        """Get quasi-diagonal order from clustering."""
        sorted_indices = []

        def append_order(node_index):
            if node_index < num_assets:
                sorted_indices.append(node_index)
            else:
                left = int(linkage_matrix[node_index - num_assets][0])
                right = int(linkage_matrix[node_index - num_assets][1])
                append_order(left)
                append_order(right)

        append_order(len(linkage_matrix) + num_assets - 1)
        return sorted_indices

    def _get_recursive_bisection(self, corr_matrix):
        """Recursive bisection for HRP weights."""
        weights = pd.Series(1.0, index=corr_matrix.index)
        clustered_alphas = [corr_matrix.index.tolist()]
        while len(clustered_alphas) > 0:
            clustered_alphas = [
                cluster[start:end]
                for cluster in clustered_alphas
                for start, end in ((0, len(cluster) // 2), (len(cluster) // 2, len(cluster)))
                if len(cluster) > 1
            ]
            for i in range(0, len(clustered_alphas), 2):
                if i + 1 < len(clustered_alphas):
                    left = clustered_alphas[i]
                    right = clustered_alphas[i + 1]
                    left_var = self._get_cluster_var(corr_matrix, left)
                    right_var = self._get_cluster_var(corr_matrix, right)
                    total_var = left_var + right_var
                    if total_var > 0:
                        left_weight = right_var / total_var
                        right_weight = left_var / total_var
                        for asset in left:
                            weights[asset] *= left_weight
                        for asset in right:
                            weights[asset] *= right_weight
        return weights.values

    def _get_cluster_var(self, corr_matrix, cluster):
        """Calculate cluster variance."""
        if len(cluster) == 1:
            return 1.0
        cluster_corr = corr_matrix.loc[cluster, cluster]
        weights = np.ones(len(cluster)) / len(cluster)
        return np.dot(weights.T, np.dot(cluster_corr.values, weights))


class BearishHedgeOptimizer:
    """
    Bearish Hedge Optimizer for defensive periods.

    ALLOCATION:
    - Small percentage to hedge assets (SH, VIXY, TLT, GLD)
    - Remainder in cash earning risk-free rate
    """

    def __init__(self, hedge_allocation: float = 0.075, risk_free_rate: float = 0.02):
        self.hedge_allocation = hedge_allocation
        self.risk_free_rate = risk_free_rate
        self.hedge_priority = ["SH", "VIXY", "TLT", "GLD"]

    def optimize(self, returns: pd.DataFrame) -> Dict[str, float]:
        """
        Create bearish hedge allocation.

        Args:
            returns: Available assets (used to check availability)

        Returns:
            Dictionary with hedge allocation
        """
        for hedge_asset in self.hedge_priority:
            if hedge_asset in returns.columns:
                return {hedge_asset: self.hedge_allocation}
        return {}


class DataManager:
    """
    Data management system for Yahoo Finance.

    FEATURES:
    - Real-time data fetching
    - Comprehensive asset universe including hedge assets
    - Data quality controls
    - No forward-filling (maintains realism)
    """

    def fetch_spy_data(self, start_date: str, end_date: str) -> pd.DataFrame:
        """
        Fetch SPY data with buffer for volatility calculations.

        Args:
            start_date: Start date (YYYY-MM-DD)
            end_date: End date (YYYY-MM-DD)

        Returns:
            DataFrame with SPY OHLCV data
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=100)).strftime("%Y-%m-%d")
        try:
            response_prices = {"historical": download_price_records("SPY", buffer_start, end_date)}
            data = response_prices
            if "historical" not in data or len(data["historical"]) < 100:
                raise ValueError(f"Insufficient SPY data: {len(data.get('historical', []))}")
            df = pd.DataFrame(data["historical"])
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            print(f"SPY data fetched: {len(df)} days (includes 100-day buffer)")
            return df[["date", "open", "high", "low", "close", "volume"]]
        except Exception as e:
            print(f"ERROR fetching SPY: {e}")
            raise

    def get_portfolio_universe(self) -> List[str]:
        """Get diversified asset universe including hedge assets."""
        return [
            "SH",
            "VIXY",
            "TLT",
            "GLD",
            "SPY",
            "QQQ",
            "IWM",
            "DIA",
            "VTI",
            "VOO",
            "IVV",
            "XLF",
            "XLE",
            "XLK",
            "XLV",
            "XLI",
            "XLY",
            "XLP",
            "XLU",
            "XLRE",
            "XLB",
            "AAPL",
            "MSFT",
            "GOOGL",
            "AMZN",
            "META",
            "NVDA",
            "EFA",
            "VEA",
            "EWJ",
            "EEM",
            "VWO",
            "IEF",
            "AGG",
            "LQD",
            "HYG",
            "VNQ",
            "JPM",
            "BAC",
            "JNJ",
            "PG",
            "WMT",
            "XOM",
            "CVX",
            "URTH",
        ]

    def fetch_portfolio_data(
        self, symbols: List[str], start_date: str, end_date: str
    ) -> Dict[str, pd.DataFrame]:
        """
        Fetch portfolio data with quality controls.

        Args:
            symbols: List of ticker symbols
            start_date: Start date
            end_date: End date

        Returns:
            Dictionary mapping symbols to DataFrames
        """
        buffer_start = (pd.to_datetime(start_date) - timedelta(days=100)).strftime("%Y-%m-%d")
        asset_data = {}
        print(f"\nFetching portfolio data for {len(symbols)} assets...")
        successful = 0
        failed = 0
        min_required_days = (pd.to_datetime(end_date) - pd.to_datetime(start_date)).days * 0.5
        for symbol in symbols:
            try:
                for attempt in range(3):
                    try:
                        response_prices = {
                            "historical": download_price_records(symbol, buffer_start, end_date)
                        }
                        data = response_prices
                        break
                    except:
                        if attempt == 2:
                            raise
                        continue
                if "historical" in data and len(data["historical"]) > min_required_days:
                    df = pd.DataFrame(data["historical"])
                    df["date"] = pd.to_datetime(df["date"])
                    df = df.sort_values("date").reset_index(drop=True)
                    df.columns = [col.lower() for col in df.columns]
                    required_cols = ["date", "open", "high", "low", "close", "volume"]
                    if all((col in df.columns for col in required_cols)):
                        df = df.dropna(subset=["open", "close"])
                        if len(df) > min_required_days:
                            asset_data[symbol] = df[required_cols].copy()
                            successful += 1
                            if symbol in ["SH", "VIXY", "TLT", "GLD"]:
                                print(f"  {symbol}: {len(df)} days (HEDGE ASSET)")
                            elif successful <= 5:
                                print(f"  {symbol}: {len(df)} days")
                else:
                    failed += 1
            except:
                failed += 1
                continue
        print(f"\nPortfolio data: {successful} successful, {failed} failed")
        hedge_available = [a for a in ["SH", "VIXY", "TLT", "GLD"] if a in asset_data]
        if hedge_available:
            print(f"Hedge assets available: {', '.join(hedge_available)}")
        else:
            print("WARNING: No hedge assets available")
        return asset_data


class WalkForwardVolatilityRegimeStrategy:
    """
    WALK-FORWARD VOLATILITY REGIME STRATEGY - PRODUCTION READY
    ==========================================================

    NO LOOK-AHEAD BIAS GUARANTEE:
    - Regime thresholds recalculated using expanding windows
    - Portfolio optimization uses only historical data
    - All predictions use T-1 data only
    - Comprehensive fold tracking for validation

    PRODUCTION FEATURES:
    - Configurable parameters
    - Realistic transaction costs
    - Professional error handling
    - Extensive analytics
    """

    def __init__(
        self,
        vol_lookback: int = 20,
        min_training_days: int = 252,
        refit_frequency: int = 63,
        rebalance_frequency: int = 21,
        lookback_window: int = 252,
        max_position_size: float = 0.15,
        hedge_allocation: float = 0.075,
        commission_rate: float = 0.0005,
        bid_ask_spread: float = 0.0003,
        market_impact: float = 0.0002,
        min_commission: float = 1.0,
        risk_free_rate: float = 0.02,
        max_sharpe_exposure: float = 1.2,
        hrp_exposure: float = 0.8,
        benchmark_symbol: str = "URTH",
        verbose: bool = True,
    ):
        """
        Initialize walk-forward strategy.

        All parameters configurable for different market conditions.
        """
        self.vol_lookback = vol_lookback
        self.min_training_days = min_training_days
        self.refit_frequency = refit_frequency
        self.rebalance_frequency = rebalance_frequency
        self.lookback_window = lookback_window
        self.max_position_size = max_position_size
        self.hedge_allocation = hedge_allocation
        self.commission_rate = commission_rate
        self.bid_ask_spread = bid_ask_spread
        self.market_impact = market_impact
        self.min_commission = min_commission
        self.risk_free_rate = risk_free_rate
        self.daily_rf_rate = risk_free_rate / 252
        self.max_sharpe_exposure = max_sharpe_exposure
        self.hrp_exposure = hrp_exposure
        self.benchmark_symbol = benchmark_symbol
        self.verbose = verbose
        self.regime_detector = WalkForwardRegimeDetector(
            vol_lookback, min_training_days, refit_frequency
        )
        self.max_sharpe_optimizer = MaxSharpeOptimizer(risk_free_rate, max_sharpe_exposure)
        self.hrp_optimizer = HRPOptimizer(target_exposure=hrp_exposure)
        self.bearish_hedge_optimizer = BearishHedgeOptimizer(hedge_allocation, risk_free_rate)
        self.data_manager = DataManager()
        self.regime_mapping = {
            0: ("HRP", "Low Vol - Downtrend"),
            1: ("MaxSharpe", "Low Vol - Uptrend"),
            2: ("BEARISH_HEDGE", "Normal Vol - Downtrend"),
            3: ("HRP", "Normal Vol - Uptrend"),
            4: ("BEARISH_HEDGE", "High Vol - Downtrend"),
            5: ("MaxSharpe", "High Vol - Uptrend"),
        }
        self.backtest_results = None

    def calculate_transaction_costs(self, trade_value: float) -> float:
        """Calculate realistic transaction costs."""
        abs_value = abs(trade_value)
        commission = max(self.commission_rate * abs_value, self.min_commission)
        spread = abs_value * self.bid_ask_spread
        impact = abs_value * self.market_impact
        return commission + spread + impact

    def optimize_by_regime(self, regime_idx: int, returns: pd.DataFrame) -> Dict[str, float]:
        """Apply regime-specific optimization."""
        if len(returns) < 60:
            return {}
        method, _ = self.regime_mapping[regime_idx]
        try:
            if method == "MaxSharpe":
                return self.max_sharpe_optimizer.optimize(returns, self.max_position_size)
            elif method == "HRP":
                return self.hrp_optimizer.optimize(returns)
            elif method == "BEARISH_HEDGE":
                return self.bearish_hedge_optimizer.optimize(returns)
        except:
            pass
        return {}

    def run_backtest(
        self,
        start_date: str,
        end_date: str,
        initial_capital: float = 100000.0,
        max_assets: int = 30,
    ) -> Dict:
        """
        Run walk-forward backtest with NO LOOK-AHEAD BIAS.

        Args:
            start_date: Backtest start (YYYY-MM-DD)
            end_date: Backtest end (YYYY-MM-DD)
            initial_capital: Starting capital
            max_assets: Maximum assets in universe

        Returns:
            Comprehensive results dictionary
        """
        if self.verbose:
            print("=" * 80)
            print("WALK-FORWARD VOLATILITY REGIME STRATEGY")
            print("=" * 80)
            print(f"Period: {start_date} to {end_date}")
            print(f"Initial Capital: ${initial_capital:,.0f}")
            print(f"Regime Refit: {self.refit_frequency} days")
            print(f"Portfolio Rebalance: {self.rebalance_frequency} days")
        print("\n1. FETCHING DATA...")
        spy_data = self.data_manager.fetch_spy_data(start_date, end_date)
        universe = self.data_manager.get_portfolio_universe()[:max_assets]
        if self.benchmark_symbol not in universe:
            universe.append(self.benchmark_symbol)
        asset_data = self.data_manager.fetch_portfolio_data(universe, start_date, end_date)
        if len(asset_data) < 5:
            raise ValueError(f"Insufficient assets: {len(asset_data)}")
        print("\n2. WALK-FORWARD REGIME DETECTION...")
        spy_features = self.regime_detector.calculate_features_t_minus_1(spy_data)
        regime_predictions, fold_list = self.regime_detector.walk_forward_predict(spy_features)
        backtest_start = pd.to_datetime(start_date)
        backtest_end = pd.to_datetime(end_date)
        backtest_data = (
            regime_predictions[
                (regime_predictions["date"] >= backtest_start)
                & (regime_predictions["date"] <= backtest_end)
            ]
            .copy()
            .reset_index(drop=True)
        )
        if len(backtest_data) < 100:
            raise ValueError(f"Insufficient backtest data: {len(backtest_data)}")
        print("\n3. PREPARING RETURNS MATRIX...")
        available_assets = [a for a in universe if a in asset_data]
        returns_dict = {}
        for asset in available_assets:
            df = asset_data[asset].copy()
            df["date"] = pd.to_datetime(df["date"])
            df = df.sort_values("date").reset_index(drop=True)
            date_range = pd.DataFrame({"date": backtest_data["date"]})
            merged = pd.merge(date_range, df[["date", "close"]], on="date", how="left")
            merged["close"] = merged["close"].ffill(limit=5)
            returns = merged["close"].pct_change()
            valid_pct = returns.notna().sum() / len(returns)
            if valid_pct >= 0.8:
                returns_dict[asset] = returns.values
        returns_matrix = pd.DataFrame(returns_dict, index=backtest_data["date"])
        returns_matrix = returns_matrix.dropna(how="all")
        available_assets = list(returns_matrix.columns)
        print(f"Clean assets: {len(available_assets)}")
        if len(available_assets) < 5:
            print(f"\nERROR: Only {len(available_assets)} assets with sufficient data")
            print("Available assets:", available_assets)
            print("\nData availability by asset:")
            for asset in universe[:10]:
                if asset in asset_data:
                    df = asset_data[asset]
                    print(
                        f"  {asset}: {len(df)} days, date range: {df['date'].min().date()} to {df['date'].max().date()}"
                    )
            raise ValueError(
                f"Insufficient clean assets: {len(available_assets)}. Need at least 5."
            )
        print(f"\n4. EXECUTING BACKTEST ({len(backtest_data)} days)...")
        portfolio_value = initial_capital
        cash = initial_capital
        positions = {a: 0.0 for a in available_assets}
        portfolio_history = []
        trade_history = []
        last_rebalance = -self.rebalance_frequency
        total_costs = 0.0
        for day_idx, row in backtest_data.iterrows():
            current_date = row["date"]
            if self.verbose and day_idx % 50 == 0:
                print(
                    f"   Day {day_idx + 1}/{len(backtest_data)}: {current_date.strftime('%Y-%m-%d')}"
                )
            regime_idx = int(row["regime_idx"]) if not pd.isna(row["regime_idx"]) else 3
            regime_name = row["regime_name"] if row["regime_name"] else "Normal Vol - Uptrend"
            current_prices = {}
            for asset in available_assets:
                if asset not in asset_data:
                    continue
                df = asset_data[asset]
                matches = df[df["date"].dt.date == current_date.date()]
                if not matches.empty:
                    current_prices[asset] = matches.iloc[0]["close"]
                else:
                    recent_data = df[df["date"] < current_date]
                    if not recent_data.empty:
                        recent_data = recent_data.tail(1)
                        days_ago = (current_date - recent_data.iloc[0]["date"]).days
                        if days_ago <= 5:
                            current_prices[asset] = recent_data.iloc[0]["close"]
            risky_value = sum((positions[a] * current_prices.get(a, 0) for a in available_assets))
            if day_idx > 0:
                cash += cash * self.daily_rf_rate
            portfolio_value = cash + risky_value
            days_since_rebal = day_idx - last_rebalance
            should_rebalance = days_since_rebal >= self.rebalance_frequency
            if (
                should_rebalance
                and day_idx >= self.lookback_window
                and (day_idx < len(backtest_data) - 1)
            ):
                hist_start = max(0, day_idx - self.lookback_window)
                hist_returns = returns_matrix.iloc[hist_start:day_idx]
                target_weights = self.optimize_by_regime(regime_idx, hist_returns)
                next_idx = day_idx + 1
                if next_idx < len(backtest_data):
                    next_date = backtest_data.iloc[next_idx]["date"]
                    exec_prices = {}
                    for asset in available_assets:
                        df = asset_data[asset]
                        matches = df[df["date"].dt.date == next_date.date()]
                        if not matches.empty:
                            exec_prices[asset] = matches.iloc[0]["open"]
                    trade_costs = 0.0
                    method, _ = self.regime_mapping[regime_idx]
                    if method in ["CASH", "BEARISH_HEDGE"] and (not target_weights):
                        for asset in available_assets:
                            if positions[asset] != 0 and asset in exec_prices:
                                value = positions[asset] * exec_prices[asset]
                                cost = self.calculate_transaction_costs(abs(value))
                                cash += value - cost
                                positions[asset] = 0.0
                                trade_costs += cost
                    else:
                        for asset, weight in target_weights.items():
                            if asset in exec_prices:
                                target_val = portfolio_value * weight
                                current_val = positions.get(asset, 0) * exec_prices[asset]
                                trade_val = target_val - current_val
                                if abs(trade_val) > portfolio_value * 0.005:
                                    cost = self.calculate_transaction_costs(trade_val)
                                    shares = trade_val / exec_prices[asset]
                                    positions[asset] = positions.get(asset, 0) + shares
                                    cash -= trade_val + cost
                                    trade_costs += cost
                        for asset in available_assets:
                            if asset not in target_weights and positions[asset] != 0:
                                if asset in exec_prices:
                                    value = positions[asset] * exec_prices[asset]
                                    cost = self.calculate_transaction_costs(abs(value))
                                    cash += value - cost
                                    positions[asset] = 0.0
                                    trade_costs += cost
                    if trade_costs > 0:
                        total_costs += trade_costs
                        trade_history.append(
                            {"date": current_date, "regime": regime_name, "cost": trade_costs}
                        )
                        last_rebalance = day_idx
            portfolio_history.append(
                {
                    "date": current_date,
                    "value": portfolio_value,
                    "cash": cash,
                    "risky": risky_value,
                    "regime": regime_name,
                }
            )
        print("\n5. CALCULATING METRICS...")
        portfolio_df = pd.DataFrame(portfolio_history)
        portfolio_df["return"] = portfolio_df["value"].pct_change()
        final_value = portfolio_df["value"].iloc[-1]
        total_return = final_value / initial_capital - 1
        days = len(portfolio_df)
        annual_return = (1 + total_return) ** (252 / days) - 1 if days > 0 else 0
        returns = portfolio_df["return"].dropna()
        volatility = returns.std() * np.sqrt(252) if len(returns) > 0 else 0
        if volatility > 0.001:
            sharpe = (annual_return - self.risk_free_rate) / volatility
        else:
            sharpe = 0.0
            print(f"   WARNING: Very low volatility ({volatility:.4f}), Sharpe set to 0")
        cumulative = (1 + returns).cumprod() if len(returns) > 0 else pd.Series([1])
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min() if len(drawdown) > 0 else 0
        metrics = {
            "initial_capital": initial_capital,
            "final_value": final_value,
            "total_return": total_return,
            "annual_return": annual_return,
            "volatility": volatility,
            "sharpe_ratio": sharpe,
            "max_drawdown": max_dd,
            "total_costs": total_costs,
            "num_trades": len(trade_history),
        }
        results = {
            "portfolio_history": portfolio_df,
            "trade_history": trade_history,
            "metrics": metrics,
            "fold_history": fold_list,
            "regime_predictions": backtest_data,
            "asset_data": asset_data,
            "config": {
                "vol_lookback": self.vol_lookback,
                "min_training": self.min_training_days,
                "refit_freq": self.refit_frequency,
                "rebalance_freq": self.rebalance_frequency,
            },
        }
        self.backtest_results = results
        if self.verbose:
            print("\n" + "=" * 80)
            print("BACKTEST COMPLETE")
            print("=" * 80)
            print(f"Final Value: ${final_value:,.0f}")
            print(f"Total Return: {total_return:.1%}")
            print(f"Annual Return: {annual_return:.1%}")
            print(f"Sharpe Ratio: {sharpe:.2f}")
            print(f"Max Drawdown: {max_dd:.1%}")
            print(f"Total Costs: ${total_costs:,.0f}")
            print(f"Walk-Forward Folds: {len(fold_list)}")
            print("=" * 80)
        return results


class StrategyVisualizer:
    """
    COMPREHENSIVE VISUALIZATION MODULE FOR STRATEGY ANALYSIS
    ========================================================

    FEATURES:
    - Equity curve comparison vs benchmark
    - Rolling beta analysis (strategy sensitivity to market)
    - Rolling Sharpe ratio (risk-adjusted performance over time)
    - Regime mapping with strategy overlay
    - Monthly returns heatmap

    NO HARDCODING: All parameters configurable through constructor
    """

    def __init__(
        self,
        rolling_window: int = 63,
        risk_free_rate: float = 0.02,
        figsize_large: Tuple[int, int] = (16, 12),
        figsize_medium: Tuple[int, int] = (14, 8),
    ):
        """
        Initialize visualizer with configurable parameters.

        Args:
            rolling_window: Window for rolling metrics (default: 63 days = ~3 months)
            risk_free_rate: Annual risk-free rate for Sharpe calculation
            figsize_large: Figure size for multi-panel plots
            figsize_medium: Figure size for single plots
        """
        self.rolling_window = rolling_window
        self.risk_free_rate = risk_free_rate
        self.daily_rf_rate = risk_free_rate / 252
        self.figsize_large = figsize_large
        self.figsize_medium = figsize_medium
        self.colors = {
            "strategy": "#1f77b4",
            "benchmark": "#ff7f0e",
            "positive": "#2ca02c",
            "negative": "#d62728",
            "neutral": "#7f7f7f",
            "regime_low": "#90EE90",
            "regime_normal": "#FFD700",
            "regime_high": "#FF6B6B",
        }
        self.regime_colors = {
            "Low Vol - Downtrend": "#90EE90",
            "Low Vol - Uptrend": "#006400",
            "Normal Vol - Downtrend": "#FFD700",
            "Normal Vol - Uptrend": "#FFA500",
            "High Vol - Downtrend": "#FF6B6B",
            "High Vol - Uptrend": "#8B0000",
        }

    def prepare_benchmark_data(
        self,
        portfolio_df: pd.DataFrame,
        asset_data: Dict[str, pd.DataFrame],
        benchmark_symbol: str,
        initial_capital: float,
    ) -> pd.DataFrame:
        """
        Prepare benchmark returns for comparison.

        EXECUTION FLOW MATCHING STRATEGY:
        - Use CLOSE prices for benchmark consistency
        - Calculate Close-to-Close returns
        - Align dates with strategy portfolio

        Args:
            portfolio_df: Strategy portfolio history
            asset_data: Dictionary of asset price data
            benchmark_symbol: Benchmark ticker (e.g., 'URTH')
            initial_capital: Starting capital for normalization

        Returns:
            DataFrame with benchmark data aligned to strategy dates
        """
        if benchmark_symbol not in asset_data:
            print(f"WARNING: Benchmark {benchmark_symbol} not available")
            return None
        bench_df = asset_data[benchmark_symbol].copy()
        bench_df["date"] = pd.to_datetime(bench_df["date"])
        portfolio_dates = portfolio_df["date"].values
        benchmark_data = []
        for date in portfolio_dates:
            matches = bench_df[bench_df["date"].dt.date == pd.to_datetime(date).date()]
            if not matches.empty:
                benchmark_data.append(
                    {"date": pd.to_datetime(date), "close": matches.iloc[0]["close"]}
                )
            else:
                recent = bench_df[bench_df["date"] < pd.to_datetime(date)]
                if not recent.empty:
                    recent = recent.tail(1)
                    days_ago = (pd.to_datetime(date) - recent.iloc[0]["date"]).days
                    if days_ago <= 5:
                        benchmark_data.append(
                            {"date": pd.to_datetime(date), "close": recent.iloc[0]["close"]}
                        )
        if not benchmark_data:
            return None
        bench_aligned = pd.DataFrame(benchmark_data)
        bench_aligned["return"] = bench_aligned["close"].pct_change()
        first_close = bench_aligned["close"].iloc[0]
        bench_aligned["value"] = bench_aligned["close"] / first_close * initial_capital
        return bench_aligned

    def calculate_rolling_beta(
        self, strategy_returns: pd.Series, benchmark_returns: pd.Series, window: int
    ) -> pd.Series:
        """
        Calculate rolling beta (strategy sensitivity to benchmark).

        Beta interpretation:
        - Beta = 1.0: Strategy moves in line with benchmark
        - Beta > 1.0: Strategy amplifies benchmark moves (higher risk)
        - Beta < 1.0: Strategy dampens benchmark moves (lower risk)
        - Beta < 0.0: Strategy moves opposite to benchmark

        Formula: Beta = Cov(strategy, benchmark) / Var(benchmark)

        Args:
            strategy_returns: Strategy return series
            benchmark_returns: Benchmark return series
            window: Rolling window size

        Returns:
            Rolling beta series
        """
        rolling_cov = strategy_returns.rolling(window).cov(benchmark_returns)
        rolling_var = benchmark_returns.rolling(window).var()
        rolling_beta = rolling_cov / rolling_var
        rolling_beta = rolling_beta.replace([np.inf, -np.inf], np.nan)
        return rolling_beta

    def calculate_rolling_sharpe(self, returns: pd.Series, window: int) -> pd.Series:
        """
        Calculate rolling Sharpe ratio.

        Sharpe measures risk-adjusted returns:
        - Higher Sharpe = Better risk-adjusted performance
        - Sharpe > 1.0: Good
        - Sharpe > 2.0: Very good
        - Sharpe > 3.0: Excellent

        Formula: Sharpe = (Return - RiskFree) / Volatility

        Args:
            returns: Return series
            window: Rolling window size

        Returns:
            Rolling Sharpe ratio series
        """
        rolling_mean = returns.rolling(window).mean() * 252
        rolling_std = returns.rolling(window).std() * np.sqrt(252)
        rolling_sharpe = (rolling_mean - self.risk_free_rate) / rolling_std
        rolling_sharpe = rolling_sharpe.replace([np.inf, -np.inf], np.nan)
        return rolling_sharpe

    def create_monthly_returns_heatmap(
        self, portfolio_df: pd.DataFrame, title: str = "Strategy Monthly Returns Heatmap"
    ) -> plt.Figure:
        """
        Create monthly returns heatmap.

        Visual representation of returns by month and year:
        - Green: Positive returns
        - Red: Negative returns
        - Color intensity: Magnitude of returns

        Args:
            portfolio_df: Portfolio history with dates and returns
            title: Chart title

        Returns:
            Matplotlib figure
        """
        df = portfolio_df.copy()
        df["date"] = pd.to_datetime(df["date"])
        df["year"] = df["date"].dt.year
        df["month"] = df["date"].dt.month
        monthly_returns = (
            df.groupby(["year", "month"])
            .apply(lambda x: (1 + x["return"]).prod() - 1)
            .reset_index(name="monthly_return")
        )
        heatmap_data = monthly_returns.pivot(index="year", columns="month", values="monthly_return")
        month_names = [
            "Jan",
            "Feb",
            "Mar",
            "Apr",
            "May",
            "Jun",
            "Jul",
            "Aug",
            "Sep",
            "Oct",
            "Nov",
            "Dec",
        ]
        heatmap_data.columns = [month_names[int(m) - 1] for m in heatmap_data.columns]
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.heatmap(
            heatmap_data * 100,
            annot=True,
            fmt=".1f",
            cmap="RdYlGn",
            center=0,
            cbar_kws={"label": "Monthly Return (%)"},
            linewidths=0.5,
            linecolor="gray",
            ax=ax,
        )
        ax.set_title(title, fontsize=14, fontweight="bold", pad=15)
        ax.set_xlabel("Month", fontsize=12)
        ax.set_ylabel("Year", fontsize=12)
        plt.tight_layout()
        return fig

    def plot_comprehensive_analysis(
        self, results: Dict, initial_capital: float, benchmark_symbol: str = "URTH"
    ) -> None:
        """
        Create comprehensive analysis with all visualizations.

        PANELS:
        1. Equity Curve: Strategy vs Benchmark
        2. Rolling Beta: Market sensitivity over time
        3. Rolling Sharpe: Risk-adjusted performance evolution
        4. Regime Mapping: Volatility regimes with price overlay

        Args:
            results: Backtest results dictionary
            initial_capital: Starting capital
            benchmark_symbol: Benchmark ticker for comparison
        """
        portfolio_df = results["portfolio_history"]
        asset_data = results["asset_data"]
        regime_data = results["regime_predictions"]
        benchmark_df = self.prepare_benchmark_data(
            portfolio_df, asset_data, benchmark_symbol, initial_capital
        )
        if benchmark_df is None:
            print(f"Cannot create comparison plots: {benchmark_symbol} data unavailable")
            return
        fig = plt.figure(figsize=self.figsize_large)
        gs = fig.add_gridspec(4, 1, hspace=0.3)
        ax1 = fig.add_subplot(gs[0])
        ax1.plot(
            portfolio_df["date"],
            portfolio_df["value"],
            label="Strategy",
            color=self.colors["strategy"],
            linewidth=2.5,
            zorder=3,
        )
        ax1.plot(
            benchmark_df["date"],
            benchmark_df["value"],
            label=f"{benchmark_symbol} (Benchmark)",
            color=self.colors["benchmark"],
            linewidth=2,
            alpha=0.8,
            zorder=2,
        )
        ax1.set_title("Equity Curve: Strategy vs Benchmark", fontsize=14, fontweight="bold")
        ax1.set_ylabel("Portfolio Value ($)", fontsize=11)
        ax1.legend(loc="upper left", fontsize=10)
        ax1.grid(True, alpha=0.3)
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x:,.0f}"))
        final_strategy = portfolio_df["value"].iloc[-1]
        final_benchmark = benchmark_df["value"].iloc[-1]
        strategy_return = (final_strategy / initial_capital - 1) * 100
        benchmark_return = (final_benchmark / initial_capital - 1) * 100
        textstr = f"Strategy: {strategy_return:+.1f}%\n{benchmark_symbol}: {benchmark_return:+.1f}%\nOutperformance: {strategy_return - benchmark_return:+.1f}%"
        ax1.text(
            0.98,
            0.02,
            textstr,
            transform=ax1.transAxes,
            fontsize=9,
            verticalalignment="bottom",
            horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5),
        )
        ax2 = fig.add_subplot(gs[1])
        strategy_returns = portfolio_df["return"].dropna()
        benchmark_returns = benchmark_df["return"].dropna()
        aligned_dates = portfolio_df.loc[strategy_returns.index, "date"]
        rolling_beta = self.calculate_rolling_beta(
            strategy_returns, benchmark_returns, self.rolling_window
        )
        ax2.plot(
            aligned_dates,
            rolling_beta,
            color=self.colors["strategy"],
            linewidth=2,
            label=f"Rolling Beta ({self.rolling_window}d)",
        )
        ax2.axhline(
            y=1.0, color="black", linestyle="--", linewidth=1.5, alpha=0.7, label="Beta = 1.0"
        )
        ax2.axhline(y=0.0, color="gray", linestyle=":", linewidth=1, alpha=0.5)
        ax2.set_title(
            f"Rolling Beta ({self.rolling_window}-day): Strategy Sensitivity to {benchmark_symbol}",
            fontsize=14,
            fontweight="bold",
        )
        ax2.set_ylabel("Beta", fontsize=11)
        ax2.legend(loc="upper left", fontsize=10)
        ax2.grid(True, alpha=0.3)
        beta_guide = "Beta > 1: Amplifies market\nBeta = 1: Tracks market\nBeta < 1: Dampens market\nBeta < 0: Inverse to market"
        ax2.text(
            0.98,
            0.98,
            beta_guide,
            transform=ax2.transAxes,
            fontsize=8,
            verticalalignment="top",
            horizontalalignment="right",
            bbox=dict(boxstyle="round", facecolor="lightblue", alpha=0.3),
        )
        ax3 = fig.add_subplot(gs[2])
        rolling_sharpe_strategy = self.calculate_rolling_sharpe(
            strategy_returns, self.rolling_window
        )
        rolling_sharpe_benchmark = self.calculate_rolling_sharpe(
            benchmark_returns, self.rolling_window
        )
        ax3.plot(
            aligned_dates,
            rolling_sharpe_strategy,
            color=self.colors["strategy"],
            linewidth=2.5,
            label="Strategy",
            zorder=3,
        )
        ax3.plot(
            aligned_dates,
            rolling_sharpe_benchmark,
            color=self.colors["benchmark"],
            linewidth=2,
            alpha=0.8,
            label=f"{benchmark_symbol}",
            zorder=2,
        )
        ax3.axhline(y=0, color="black", linestyle="-", linewidth=1, alpha=0.5)
        ax3.axhline(y=1, color="green", linestyle="--", linewidth=1, alpha=0.3, label="Good (>1.0)")
        ax3.axhline(
            y=2, color="darkgreen", linestyle="--", linewidth=1, alpha=0.3, label="Very Good (>2.0)"
        )
        ax3.set_title(
            f"Rolling Sharpe Ratio ({self.rolling_window}-day): Risk-Adjusted Performance",
            fontsize=14,
            fontweight="bold",
        )
        ax3.set_ylabel("Sharpe Ratio", fontsize=11)
        ax3.legend(loc="upper left", fontsize=9, ncol=2)
        ax3.grid(True, alpha=0.3)
        ax4 = fig.add_subplot(gs[3])
        if "SPY" in asset_data:
            spy_df = asset_data["SPY"].copy()
            spy_df["date"] = pd.to_datetime(spy_df["date"])
            regime_dates = regime_data["date"]
            spy_aligned = []
            for date in regime_dates:
                matches = spy_df[spy_df["date"].dt.date == pd.to_datetime(date).date()]
                if not matches.empty:
                    spy_aligned.append(
                        {"date": pd.to_datetime(date), "close": matches.iloc[0]["close"]}
                    )
            if spy_aligned:
                spy_plot = pd.DataFrame(spy_aligned)
                ax4_twin = ax4.twinx()
                ax4_twin.plot(
                    spy_plot["date"],
                    spy_plot["close"],
                    color="black",
                    linewidth=1,
                    alpha=0.3,
                    label="SPY Price",
                )
                ax4_twin.set_ylabel("SPY Price ($)", fontsize=11)
                ax4_twin.legend(loc="upper right", fontsize=9)
        regime_dates = regime_data["date"]
        regime_names = regime_data["regime_name"]
        for i in range(len(regime_dates) - 1):
            regime = regime_names.iloc[i]
            if pd.notna(regime) and regime in self.regime_colors:
                ax4.axvspan(
                    regime_dates.iloc[i],
                    regime_dates.iloc[i + 1],
                    alpha=0.3,
                    color=self.regime_colors[regime],
                )
        legend_patches = [
            mpatches.Patch(color=color, label=regime, alpha=0.5)
            for regime, color in self.regime_colors.items()
        ]
        ax4.legend(handles=legend_patches, loc="upper left", fontsize=8, ncol=2)
        ax4.set_title("Volatility Regime Mapping (SPY)", fontsize=14, fontweight="bold")
        ax4.set_xlabel("Date", fontsize=11)
        ax4.set_ylabel("Regime", fontsize=11)
        ax4.set_yticks([])
        ax4.grid(True, alpha=0.3, axis="x")
        plt.tight_layout()
        plt.show()
        print("\n" + "=" * 80)
        print("VISUALIZATION COMPLETE")
        print("=" * 80)

    def plot_monthly_heatmap(self, results: Dict) -> None:
        """
        Create standalone monthly returns heatmap.

        Args:
            results: Backtest results dictionary
        """
        portfolio_df = results["portfolio_history"]
        fig = self.create_monthly_returns_heatmap(portfolio_df)
        plt.show()


def main():
    """Execute walk-forward strategy with comprehensive visualizations."""
    config = {
        "vol_lookback": 20,
        "min_training_days": 252,
        "refit_frequency": 63,
        "rebalance_frequency": 21,
        "lookback_window": 252,
        "max_position_size": 0.15,
        "hedge_allocation": 0.075,
        "commission_rate": 0.0005,
        "bid_ask_spread": 0.0003,
        "market_impact": 0.0002,
        "min_commission": 1.0,
        "risk_free_rate": 0.02,
        "max_sharpe_exposure": 1.2,
        "hrp_exposure": 0.8,
        "benchmark_symbol": "URTH",
        "verbose": True,
    }
    print("=" * 80)
    print("WALK-FORWARD VOLATILITY REGIME STRATEGY - PRODUCTION VERSION")
    print("=" * 80)
    strategy = WalkForwardVolatilityRegimeStrategy(**config)
    results = strategy.run_backtest(
        start_date="2013-01-01", end_date="2018-12-31", initial_capital=100000.0, max_assets=30
    )
    print("\n" + "=" * 80)
    print("GENERATING COMPREHENSIVE VISUALIZATIONS")
    print("=" * 80)
    visualizer = StrategyVisualizer(
        rolling_window=63, risk_free_rate=0.02, figsize_large=(16, 12), figsize_medium=(14, 8)
    )
    visualizer.plot_comprehensive_analysis(
        results=results, initial_capital=100000.0, benchmark_symbol=config["benchmark_symbol"]
    )
    print("\nGenerating monthly returns heatmap...")
    visualizer.plot_monthly_heatmap(results)
    return (strategy, results, visualizer)


if __name__ == "__main__":
    strategy, results, visualizer = main()
